# 01 · Per-section pipeline — BANKSY

For every `.h5ad` section in `PROJ_DIR`:
1. Step 1 QC → auto-suggest `k_geom` per section
2. Preprocess: normalize → log1p, store all layers + raw counts
3. BANKSY per section → 1500-feature matrix (original genes + spatial neighbourhood)
4. Save two h5ad files per section

```
SAVE_DIR/<section>/
  <section>_banksy_full.h5ad   <- 1500 features (genes + neighbourhood)
  <section>_banksy_genes.h5ad  <- genes only, all layers, raw = counts
```

## 1 · Imports

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from sklearn.neighbors import NearestNeighbors
from banksy.initialize_banksy import initialize_banksy
from banksy.embed_banksy import generate_banksy_matrix
from banksy.main import median_dist_to_nearest_neighbour
warnings.filterwarnings("ignore")
sc.logging.print_header()

## 2 · Paths & parameters


In [ ]:
PROJ_DIR = "/path/to/your/input/h5ad/files/"
SAVE_DIR = "/path/to/your/output/directory/"

# k_geom per section — auto-filled by Step 1, override manually if needed
K_GEOM_PER_SECTION = {}
DEFAULT_K_GEOM     = 8
K_GEOM_CANDIDATES  = [6, 8, 10, 12]

PARAMS = dict(
    seed             = 1234,
    max_m            = 1,
    lambda_list      = [0.8],
    nbr_weight_decay = "scaled_gaussian",
    k_geom = 8
)


## 3 · Helper functions

### 3.1 · Environment

In [ ]:
def setup_environment(seed):
    np.random.seed(seed); random.seed(seed)
    sc.set_figure_params(facecolor="white", figsize=(8, 8))
    sc.settings.verbosity = 1
    sns.set_style("white")

### 3.2 · Step 1 — k_geom QC

In [ ]:
def compute_knn_distances(h5ad_files, k=12):
    """Compute median k-NN distances per section to guide k_geom selection."""
    results = []
    for path in h5ad_files:
        name  = os.path.splitext(os.path.basename(path))[0]
        adata = sc.read_h5ad(path)
        if adata.n_obs < k:
            continue
        coords  = np.vstack([adata.obs["x"], adata.obs["y"]]).T
        nbrs    = NearestNeighbors(n_neighbors=k).fit(coords)
        dist, _ = nbrs.kneighbors(coords)
        dist    = dist[:, 1:]
        results.append({
            "section"   : name,
            "median_k7" : np.median(dist[:, 6]),
            "median_k8" : np.median(dist[:, 7]),
            "median_k10": np.median(dist[:, 9]),
            "n_cells"   : adata.n_obs,
        })
    return pd.DataFrame(results).sort_values("median_k8")


### 3.3 · Preprocessing

In [ ]:
def preprocess(adata):
    """
    Store raw counts, normalize, log-transform.

    Layers: counts | normalized | transformed | scaled
    adata.raw = counts   (frozen for DE tools)
    adata.X   = transformed  (log-normalized — input for BANKSY)
    """
    if not sp.issparse(adata.X):
        adata.X = sp.csr_matrix(adata.X) 

    adata.layers["counts"] = adata.X.copy()
    adata.raw = adata
    
    sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)
    adata.layers["normalized"] = adata.X.copy()

    sc.pp.log1p(adata)
    adata.layers["transformed"] = adata.X.copy()
    adata.layers["scaled"]      = adata.X.copy()

    adata.X = adata.layers["transformed"].copy()
    if not sp.isspmatrix_csr(adata.X):
        adata.X = adata.X.tocsr()
    adata.X = adata.X.astype(np.float32)

    print(f"        Layers: {list(adata.layers.keys())}")
    print(f"        adata.X = transformed (log-normalized) — ready for BANKSY")
    print(f"Found {len(h5ad_files)} sections\n")

    return adata

### 3.4 · BANKSY

In [ ]:
def run_banksy(adata, k_geom):
    """
    Run BANKSY on adata.X (transformed) using local section coordinates.
    Returns (banksy_dict, adata_joint) — 1500-feature BANKSY matrix.
    """
    coord_keys = ("x", "y", "coord_xy")
    adata.obsm["coord_xy"] = np.vstack([
        adata.obs["x"].to_numpy(),
        adata.obs["y"].to_numpy()
    ]).T.astype(np.float32)

    _ = median_dist_to_nearest_neighbour(adata, key="coord_xy")

    banksy_dict = initialize_banksy(
        adata, coord_keys, k_geom,
        nbr_weight_decay = PARAMS["nbr_weight_decay"],
        max_m            = PARAMS["max_m"],
        plt_edge_hist=False, plt_nbr_weights=False,
        plt_agf_angles=False, plt_theta=False,
    )
    banksy_dict, _ = generate_banksy_matrix(
        adata, banksy_dict, PARAMS["lambda_list"], PARAMS["max_m"]
    )
    lam         = float(PARAMS["lambda_list"][0])
    adata_joint = banksy_dict[PARAMS["nbr_weight_decay"]][lam]["adata"].copy()
    print(f"        BANKSY matrix: {adata_joint.shape[1]:,} features")
    return banksy_dict, adata_joint

### 3.5 · Save

In [ ]:
def save_outputs(adata_orig, adata_joint, section_name, section_dir):
    """
    Save:
    1. <section>_banksy_full.h5ad  — 1500 features (genes + neighbourhood)
    2. <section>_banksy_genes.h5ad — genes only, all layers, raw = counts
    """
    path_full = os.path.join(section_dir, f"{section_name}_banksy_full.h5ad")
    adata_joint.write(path_full)
    print(f"        [✓] Full h5ad  ({adata_joint.shape[1]:,} features) → {path_full}")

    genes_keep    = adata_orig.var_names.intersection(adata_joint.var_names)
    adata_genes   = adata_joint[:, genes_keep].copy()
    adata_aligned = adata_orig[adata_genes.obs_names, adata_genes.var_names]

    for layer in ["counts", "normalized", "transformed", "scaled"]:
        if layer in adata_aligned.layers:
            adata_genes.layers[layer] = adata_aligned.layers[layer].copy()

    adata_genes.X = adata_genes.layers["transformed"].copy()
    if sp.issparse(adata_genes.X):
        adata_genes.X = adata_genes.X.tocsr()
    adata_genes.raw = sc.AnnData(
        X=adata_genes.layers["counts"],
        obs=adata_genes.obs.copy(),
        var=adata_genes.var.copy(),
    )
    path_genes = os.path.join(section_dir, f"{section_name}_banksy_genes.h5ad")
    adata_genes.write(path_genes)
    print(f"        [✓] Genes h5ad ({adata_genes.shape[1]:,} genes)    → {path_genes}")

### 3.6 · Per-section orchestrator

In [ ]:
def process_section(h5ad_path, save_dir):
    section_name = os.path.splitext(os.path.basename(h5ad_path))[0]
    section_dir  = os.path.join(save_dir, section_name)
    k_geom = DEFAULT_K_GEOM
    os.makedirs(section_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  Section : {section_name}  |  k_geom = {k_geom} "
          f"({'auto' if section_name in K_GEOM_PER_SECTION else 'default'})")
    print(f"{'='*60}")

    print("  [1/3] Loading...")
    adata = sc.read_h5ad(h5ad_path)
    print(f"        {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")

    print("  [2/3] Preprocessing...")
    adata = preprocess(adata)

    print(f"  [3/3] BANKSY (k_geom={k_geom})...")
    _, adata_joint = run_banksy(adata, k_geom)

    print("  Saving...")
    save_outputs(adata, adata_joint, section_name, section_dir)
    print(f"  Done — {section_name}!")

## 4 · Step 1 — Choose k_geom

Run this first, inspect the table, then the auto-suggestion is applied automatically.

In [ ]:
setup_environment(PARAMS["seed"])

h5ad_files = sorted(
    os.path.join(PROJ_DIR, f)
    for f in os.listdir(PROJ_DIR)
    if f.endswith(".h5ad") and not f.startswith("._")
)

df_knn = compute_knn_distances(h5ad_files)
print("Median k-NN distances per section (sorted by median_k8):")
print(df_knn.to_string(index=False))

**→ Override any section in `K_GEOM_PER_SECTION` if needed, then run Step 2.**

## 5 · Step 2 — Run all sections

In [ ]:
print(f"Sections to process:")
for f in h5ad_files:
    sec = os.path.splitext(os.path.basename(f))[0]
    print(f"  {sec}  (k_geom={K_GEOM_PER_SECTION.get(sec, DEFAULT_K_GEOM)})")

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

for h5ad_path in h5ad_files:
    try: 
        process_section(h5ad_path, SAVE_DIR)
    except Exception as e:
        print(f"  [!] ERROR {os.path.basename(h5ad_path)}: {e}")
        continue

print("\nAll sections processed.")